In [1]:
import sys
sys.path.append("..")          # so Python can find the src folder

import pandas as pd
import numpy as np
from src.features import build_features

df = pd.read_csv("../data/clean.csv")
date_cols = df.columns[1:-2]
usage = df[date_cols].copy()
usage.columns = pd.to_datetime(usage.columns)
print(usage.shape)

(42289, 1034)


In [2]:
features = build_features(usage)
features["MISSING_PCT"] = df["MISSING_PCT"]
features.insert(0, "CONS_NO", df["CONS_NO"])
features["FLAG"] = df["FLAG"]

print(features.shape)
features.head()

(42289, 21)


,CONS_NO,mean,median,std,max,cv,start_day,active_days,zero_days_after_start,zero_pct_after_start,...,longest_zero_streak,max_month_drop,big_drops_count,monthly_std,recent_vs_overall,change_2016_vs_2015,summer_vs_winter,weekend_vs_weekday,MISSING_PCT,FLAG
0,0387DD8A07E07FDA6271170F86AD9151,0.8182,0.0000,2.5587,10.2578,3.1271,2,1032,863,0.8362,...,553,1.0000,1,2.1669,0.1164,0.1543,0.0115,1.0432,0.39,1
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,4.1698,1.2811,6.2425,24.9034,1.4971,50,984,0,0.0000,...,0,0.4477,0,6.0823,3.9089,10.0000,3.6108,1.0373,18.18,1
2,4B75AC4F2D8434CFF62DB64D0BB43103,8.4480,6.5909,7.9720,37.9552,0.9436,5,1029,29,0.0282,...,12,0.6875,2,5.9618,1.1238,0.4859,2.4295,1.0957,30.95,1
3,B32AC8CC6D5D805AC053557AB05F5343,3.6119,0.0000,7.4472,27.9599,2.0618,800,234,0,0.0000,...,0,0.3157,0,7.2086,4.9076,1.0000,1.0000,1.0759,1.64,1
4,EDFC78B07BA2908B3395C4EB2304665E,87.7282,88.5980,43.1067,226.2614,0.4914,35,999,3,0.0030,...,2,0.9217,2,38.5950,0.9259,0.8543,1.5861,1.0006,65.67,1


In [3]:
num = features.drop(columns=["CONS_NO"])
print("Missing values:", num.isnull().sum().sum())
print("Infinite values:", np.isinf(num.values).sum())

Missing values: 0
Infinite values: 0


In [4]:
from sklearn.metrics import roc_auc_score

scores = {}
for col in features.columns.drop(["CONS_NO", "FLAG"]):
    auc = roc_auc_score(features["FLAG"], features[col])
    scores[col] = round(max(auc, 1 - auc), 3)

print(pd.Series(scores).sort_values(ascending=False))

monthly_std              0.705
std                      0.700
max                      0.688
mean                     0.642
recent_vs_overall        0.601
max_month_drop           0.580
change_2016_vs_2015      0.574
median                   0.572
MISSING_PCT              0.568
big_drops_count          0.558
summer_vs_winter         0.557
cv                       0.551
zero_pct_after_start     0.543
weekend_vs_weekday       0.520
zero_streaks             0.519
active_days              0.518
start_day                0.518
zero_days_after_start    0.507
longest_zero_streak      0.504
dtype: float64


In [5]:
features.to_csv("../data/features.csv", index=False)
print("Saved!")


Saved!
